# MVP — Pipeline de Dados em Nuvem (Databricks / Delta Lake)
## Disciplina: Sistemas de Apoio à Decisão (PSP8) — Engenharia de Produção / UnB
**Aluno:** Rodrigo Fregonasse  
**Professor:** Dr. André Luiz Marques Serrano  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist (Kaggle)

---
### Objetivo do Notebook
Este notebook implementa a arquitetura de dados Medalhão (**Bronze $ightarrow$ Silver $ightarrow$ Gold**) na plataforma de nuvem **Databricks**, realizando:
1. Ingestão e persistência dos dados brutos no DBFS (*Databricks File System*)
2. Limpeza, tratamento de tipos e valores nulos com PySpark
3. Aplicação de regras de negócio de logística e engenharia de produção
4. Construção da Modelagem Dimensional em **Esquema Estrela (*Star Schema*)**
5. Persistência das tabelas finais no formato colunar de alto desempenho **Delta Lake**

In [ ]:
# 1. Configuração e Instalação de Dependências no Cluster Databricks
%pip install kagglehub pandas numpy matplotlib seaborn --quiet

### 2. Ingestão dos Dados Brutos para o Databricks (Camada Bronze)
Baixamos os microdados oficiais da Olist e salvamos no armazenamento persistente do Databricks ().

In [ ]:
import os
import shutil
import kagglehub

# Download direto via kagglehub no ambiente do driver
path = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(f'Dados baixados no driver: {path}')

# Copiar para o DBFS (armazenamento persistente em nuvem do Databricks)
dbfs_target = '/dbfs/FileStore/tables/olist'
os.makedirs(dbfs_target, exist_ok=True)

for fname in os.listdir(path):
    if fname.endswith('.csv'):
        shutil.copy(os.path.join(path, fname), os.path.join(dbfs_target, fname))
        print(f'Persistido no DBFS: {fname}')

### 3. Carga das Tabelas Bronze via PySpark
Carregamos os dados em DataFrames distribuídos do Apache Spark.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

bronze_path = '/FileStore/tables/olist/'

df_orders = spark.read.csv(bronze_path + 'olist_orders_dataset.csv', header=True, inferSchema=True)
df_items = spark.read.csv(bronze_path + 'olist_order_items_dataset.csv', header=True, inferSchema=True)
df_customers = spark.read.csv(bronze_path + 'olist_customers_dataset.csv', header=True, inferSchema=True)
df_sellers = spark.read.csv(bronze_path + 'olist_sellers_dataset.csv', header=True, inferSchema=True)
df_products = spark.read.csv(bronze_path + 'olist_products_dataset.csv', header=True, inferSchema=True)
df_reviews = spark.read.csv(bronze_path + 'olist_order_reviews_dataset.csv', header=True, inferSchema=True)
df_trans = spark.read.csv(bronze_path + 'product_category_name_translation.csv', header=True, inferSchema=True)

print(f'Pedidos brutos: {df_orders.count():,}')
print(f'Itens de pedidos: {df_items.count():,}')

### 4. Camada Silver: Limpeza, Tipagem e Deduplicação
Tratamos timestamps, removemos inconsistências físicas e preparamos os atributos essenciais.

In [ ]:
# Filtrar pedidos entregues e converter timestamps
df_orders_delivered = df_orders.filter(F.col('order_status') == 'delivered')     .filter(F.col('order_delivered_customer_date').isNotNull())     .withColumn('ts_compra', F.to_timestamp('order_purchase_timestamp'))     .withColumn('ts_aprovacao', F.to_timestamp('order_approved_at'))     .withColumn('ts_despacho', F.to_timestamp('order_delivered_carrier_date'))     .withColumn('ts_entrega_real', F.to_timestamp('order_delivered_customer_date'))     .withColumn('ts_prazo_estimado', F.to_timestamp('order_estimated_delivery_date'))

# Deduplicar avaliações (manter a mais recente por pedido)
from pyspark.sql.window import Window
w_rev = Window.partitionBy('order_id').orderBy(F.col('review_answer_timestamp').desc())
df_reviews_dedup = df_reviews.withColumn('rn', F.row_number().over(w_rev))     .filter(F.col('rn') == 1)     .select('order_id', 'review_score')

print(f'Pedidos entregues válidos na Silver: {df_orders_delivered.count():,}')

### 5. Camada Gold: Modelagem Dimensional (Esquema Estrela)
Construímos as Dimensões Conformes e a Tabela Fato com as métricas logísticas calculadas.

In [ ]:
# Dicionário de Regiões do Brasil
regioes_map = F.create_map([F.lit(x) for x in [
    'SP', 'Sudeste', 'RJ', 'Sudeste', 'MG', 'Sudeste', 'ES', 'Sudeste',
    'PR', 'Sul', 'SC', 'Sul', 'RS', 'Sul',
    'DF', 'Centro-Oeste', 'GO', 'Centro-Oeste', 'MT', 'Centro-Oeste', 'MS', 'Centro-Oeste',
    'BA', 'Nordeste', 'PE', 'Nordeste', 'CE', 'Nordeste', 'MA', 'Nordeste',
    'PB', 'Nordeste', 'RN', 'Nordeste', 'AL', 'Nordeste', 'PI', 'Nordeste', 'SE', 'Nordeste',
    'AM', 'Norte', 'PA', 'Norte', 'RO', 'Norte', 'TO', 'Norte',
    'AC', 'Norte', 'AP', 'Norte', 'RR', 'Norte'
]])

# Dimensão Clientes
dim_clientes = df_customers.select(
    F.col('customer_id').alias('cliente_id'),
    F.col('customer_unique_id').alias('cliente_hash_unico'),
    F.col('customer_zip_code_prefix').alias('cep_prefixo_cliente'),
    F.col('customer_city').alias('cidade_cliente'),
    F.col('customer_state').alias('estado_cliente'),
    regioes_map[F.col('customer_state')].alias('regiao_cliente')
)

# Dimensão Vendedores
dim_vendedores = df_sellers.select(
    F.col('seller_id').alias('vendedor_id'),
    F.col('seller_zip_code_prefix').alias('cep_prefixo_vendedor'),
    F.col('seller_city').alias('cidade_vendedor'),
    F.col('seller_state').alias('estado_vendedor'),
    regioes_map[F.col('seller_state')].alias('regiao_vendedor')
)

# Dimensão Produtos
dim_produtos = df_products.join(df_trans, on='product_category_name', how='left')     .select(
        F.col('product_id').alias('produto_id'),
        F.coalesce(F.col('product_category_name'), F.lit('outros')).alias('categoria_pt'),
        F.coalesce(F.col('product_category_name_english'), F.lit('other')).alias('categoria_en'),
        F.coalesce(F.col('product_weight_g'), F.lit(1000)).alias('peso_gramas'),
        (F.coalesce(F.col('product_length_cm'), F.lit(20)) *
         F.coalesce(F.col('product_height_cm'), F.lit(15)) *
         F.coalesce(F.col('product_width_cm'), F.lit(15))).alias('volume_cm3')
    )     .withColumn('faixa_peso', F.when(F.col('peso_gramas') < 1000, '1. Leve (<1kg)')
                .when(F.col('peso_gramas') <= 5000, '2. Médio (1-5kg)')
                .when(F.col('peso_gramas') <= 15000, '3. Pesado (5-15kg)')
                .otherwise('4. Muito Pesado (>15kg)'))

### 6. Tabela Fato: 
Unificamos os dados e calculamos métricas de Lead Time, Atraso e Satisfação.

In [ ]:
fato = df_items.join(df_orders_delivered, on='order_id', how='inner')     .join(df_customers.select('customer_id', 'customer_state'), on='customer_id', how='left')     .join(df_sellers.select('seller_id', 'seller_state'), on='seller_id', how='left')     .join(df_reviews_dedup, on='order_id', how='left')

# Cálculos em dias (segundos / 86400)
fato_entregas = fato.select(
    F.col('order_id').alias('pedido_id'),
    F.col('order_item_id').alias('item_id'),
    F.col('customer_id').alias('cliente_id'),
    F.col('seller_id').alias('vendedor_id'),
    F.col('product_id').alias('produto_id'),
    F.date_format('ts_compra', 'yyyyMMdd').cast('int').alias('data_pedido_id'),
    F.concat(F.col('seller_state'), F.lit('-'), F.col('customer_state')).alias('rota_id'),
    F.col('price').alias('valor_preco'),
    F.col('freight_value').alias('valor_frete'),
    (F.col('price') + F.col('freight_value')).alias('valor_total'),
    F.round(F.col('freight_value') / F.col('price'), 4).alias('razao_frete_preco'),
    F.round((F.col('ts_entrega_real').cast('long') - F.col('ts_compra').cast('long')) / 86400.0, 2).alias('lead_time_real_dias'),
    F.round((F.col('ts_prazo_estimado').cast('long') - F.col('ts_compra').cast('long')) / 86400.0, 2).alias('prazo_prometido_dias'),
    F.round((F.col('ts_entrega_real').cast('long') - F.col('ts_prazo_estimado').cast('long')) / 86400.0, 2).alias('dias_desvio_prazo'),
    F.col('review_score')
) .withColumn('flag_atrasado', F.when(F.col('dias_desvio_prazo') > 0, 1).otherwise(0)) .withColumn('flag_atraso_grave_48h', F.when(F.col('dias_desvio_prazo') > 2.0, 1).otherwise(0)) .withColumn('flag_detrator', F.when((F.col('review_score').isNotNull()) & (F.col('review_score') <= 2), 1).otherwise(0))

print(f'Total de registros na Fato Entregas: {fato_entregas.count():,}')

### 7. Persistência no Formato Delta Lake (Nuvem)
Gravamos as tabelas no catálogo Delta para permitir consultas SQL de alto desempenho e ACID transactions.

In [ ]:
# Salvar tabelas em formato Delta Lake
fato_entregas.write.format('delta').mode('overwrite').saveAsTable('gold_fato_entregas_pedidos')
dim_clientes.write.format('delta').mode('overwrite').saveAsTable('gold_dim_clientes')
dim_vendedores.write.format('delta').mode('overwrite').saveAsTable('gold_dim_vendedores')
dim_produtos.write.format('delta').mode('overwrite').saveAsTable('gold_dim_produtos')

print('Todas as tabelas da Camada Gold foram persistidas em Delta Lake no Databricks!')

In [ ]:
# Validação rápida via Spark SQL
%sql
SELECT 
    flag_atrasado, 
    COUNT(*) as total_itens, 
    ROUND(AVG(lead_time_real_dias), 2) as lead_time_medio,
    ROUND(AVG(review_score), 2) as review_medio,
    ROUND(AVG(flag_detrator) * 100, 2) as pct_detratores
FROM gold_fato_entregas_pedidos
GROUP BY flag_atrasado;